In [1]:
"""
TrustChainAi Vulnerability Detector Training Notebook
=====================================================

This notebook trains a CodeBERT-based model to detect vulnerabilities in smart contracts.

Pipeline:
1. Load 10,000 Ethereum contracts (from existing RawEthSmartContracts dataset)
2. Assign vulnerability labels using bytecode pattern analysis
3. Split into SolidiFI (curated) and SmartBugs (large) datasets
4. Fine-tune CodeBERT for multi-class vulnerability classification
5. Evaluate metrics (accuracy, precision, recall, F1)
6. Save model, tokenizer, and configuration for inference

Training Uses:
- Model: CodeBERT (microsoft/codebert-base) - pre-trained on code
- Tokenizer: HuggingFace AutoTokenizer
- Framework: HuggingFace Transformers + PyTorch
- Hardware: GPU if available, CPU fallback
- Training time: ~15-30 minutes on GPU, 1-2 hours on CPU

Output Artifacts:
- data/models/vulnerability_detector_v1/ (trained model)
- data/models/vulnerability_detector_v1/pytorch_model.bin (weights)
- data/models/vulnerability_detector_v1/config.json (model config + metrics)
- data/models/vulnerability_detector_v1/label_map.json (vulnerability type mapping)
"""

# === CELL 1: Environment Setup ===
# Import core libraries
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np
import os
from pathlib import Path
import json


# Print system information
print("="*60)
print("TRUSTCHAINAI TRAINING ENVIRONMENT")
print("="*60)
print(f"PyTorch version: {torch.__version__}")
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("Device: CPU (slower training, but still works)")
print("="*60)

TRUSTCHAINAI TRAINING ENVIRONMENT
PyTorch version: 2.10.0+cpu
GPU Available: False
Device: CPU (slower training, but still works)


In [2]:
# === CELL 2: Load Training Datasets ===
"""
Load SolidiFI (smaller, curated set) and SmartBugs (larger real-world set).

These datasets were prepared by scripts/prepare_training_data.py which:
1. Loaded 10,000 raw Ethereum contracts
2. Analyzed bytecode for vulnerability patterns
3. Assigned labels: safe, reentrancy, overflow, phishing
4. Split into training (7000) and validation (3000)

The datasets are stored as CSV files:
- SolidiFI: data/Datasets/SolidiFI/dataset.csv (curated set)
- SmartBugs: data/Datasets/SmartBugs/dataset.csv (large production set)
"""

# Resolve paths from project root
notebook_dir = Path.cwd()
project_root = notebook_dir.parent  # One level up from notebooks/
data_dir = project_root / 'data' / 'Datasets'

solidifi_path = data_dir / 'SolidiFI' / 'dataset.csv'
smartbugs_path = data_dir / 'SmartBugs' / 'dataset.csv'

print("\n" + "="*60)
print("STEP 1: LOADING DATASETS")
print("="*60)
print(f"Project root: {project_root}")
print(f"Looking for datasets in: {data_dir}")
print(f"  - SolidiFI: {solidifi_path}")
print(f"  - SmartBugs: {smartbugs_path}")

# Validate datasets exist
if not solidifi_path.exists() or not smartbugs_path.exists():
    missing = []
    if not solidifi_path.exists():
        missing.append(f"  ✗ SolidiFI: {solidifi_path}")
    if not smartbugs_path.exists():
        missing.append(f"  ✗ SmartBugs: {smartbugs_path}")
    
    error_msg = ("DATASETS NOT FOUND!\n\n" + 
                 "Missing datasets:\n" + 
                 "\n".join(missing) + 
                 "\n\nTo prepare datasets, run from project root:\n" +
                 "  python scripts/prepare_training_data.py\n\n" +
                 "This will convert existing Ethereum contracts to training format.")
    raise FileNotFoundError(error_msg)

# Load datasets
print("\n✓ Loading SolidiFI...")
solidifi_df = pd.read_csv(solidifi_path)
print(f"  {len(solidifi_df)} contracts loaded")

print("✓ Loading SmartBugs...")
smartbugs_df = pd.read_csv(smartbugs_path)
print(f"  {len(smartbugs_df)} contracts loaded")

print(f"\n✓ Total: {len(solidifi_df) + len(smartbugs_df)} contracts ready for training")


STEP 1: LOADING DATASETS
Project root: c:\Users\phili\Documents\Projects\TrustChainAi
Looking for datasets in: c:\Users\phili\Documents\Projects\TrustChainAi\data\Datasets
  - SolidiFI: c:\Users\phili\Documents\Projects\TrustChainAi\data\Datasets\SolidiFI\dataset.csv
  - SmartBugs: c:\Users\phili\Documents\Projects\TrustChainAi\data\Datasets\SmartBugs\dataset.csv

✓ Loading SolidiFI...
  3000 contracts loaded
✓ Loading SmartBugs...
  7000 contracts loaded

✓ Total: 10000 contracts ready for training


In [3]:
# === CELL 3: Prepare Labels & Balance Dataset ===
"""
Convert vulnerability labels and fix dataset imbalance.

Binary Classification Setup:
0 = SAFE
1 = VULNERABLE

Vulnerable includes:
- overflow
- reentrancy
- phishing

This approach is standard in vulnerability detection research because
real-world smart contract datasets are heavily imbalanced.
"""

print("\n" + "="*60)
print("STEP 2: LABEL PROCESSING & DATASET BALANCING")
print("="*60)

# Combine datasets
combined_df = pd.concat([
    solidifi_df[['code', 'vulnerability_type']],
    smartbugs_df[['code', 'vulnerability_type']]
], ignore_index=True)

print(f"\n✓ Combined dataset size: {len(combined_df)} contracts")

# Binary label mapping
def map_label(vuln):
    if vuln == "safe":
        return 0
    else:
        return 1

combined_df["label"] = combined_df["vulnerability_type"].apply(map_label)

# Show original distribution
print("\nOriginal vulnerability distribution:")
original_counts = combined_df["label"].value_counts()

for label, count in original_counts.items():
    name = "safe" if label == 0 else "vulnerable"
    pct = (count / len(combined_df)) * 100
    print(f"  {name:12} {count:6d} ({pct:5.2f}%)")

# Separate classes
safe_df = combined_df[combined_df["label"] == 0]
vuln_df = combined_df[combined_df["label"] == 1]

print(f"\nSafe contracts: {len(safe_df)}")
print(f"Vulnerable contracts: {len(vuln_df)}")

# Downsample SAFE class to reduce imbalance
target_safe = min(len(safe_df), len(vuln_df) * 5)

balanced_safe = safe_df.sample(n=target_safe, random_state=42)

balanced_df = pd.concat([balanced_safe, vuln_df])

print(f"\n✓ Balanced dataset size: {len(balanced_df)}")

# Show balanced distribution
print("\nBalanced dataset distribution:")
balanced_counts = balanced_df["label"].value_counts()

for label, count in balanced_counts.items():
    name = "safe" if label == 0 else "vulnerable"
    pct = (count / len(balanced_df)) * 100
    print(f"  {name:12} {count:6d} ({pct:5.2f}%)")

# Train / validation split
print("\n✓ Splitting dataset (80% train / 20% validation)")

train_df, val_df = train_test_split(
    balanced_df,
    test_size=0.2,
    random_state=42,
    stratify=balanced_df["label"]
)

print(f"Train set: {len(train_df)}")
print(f"Validation set: {len(val_df)}")

# Convert to HuggingFace Dataset
print("\n✓ Converting to HuggingFace Dataset format")

train_dataset = Dataset.from_pandas(train_df[['code', 'label']])
val_dataset = Dataset.from_pandas(val_df[['code', 'label']])

print(f"\n✓ Ready for training")
print(f"Train samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")


STEP 2: LABEL PROCESSING & DATASET BALANCING

✓ Combined dataset size: 10000 contracts

Original vulnerability distribution:
  safe           9959 (99.59%)
  vulnerable       41 ( 0.41%)

Safe contracts: 9959
Vulnerable contracts: 41

✓ Balanced dataset size: 246

Balanced dataset distribution:
  safe            205 (83.33%)
  vulnerable       41 (16.67%)

✓ Splitting dataset (80% train / 20% validation)
Train set: 196
Validation set: 50

✓ Converting to HuggingFace Dataset format

✓ Ready for training
Train samples: 196
Validation samples: 50


In [4]:
# === CELL 4: Load Pre-trained Model & Tokenizer ===
"""
Load CodeBERT (a transformer model pre-trained on source code) and fine-tune it
for smart contract vulnerability detection.

Why CodeBERT?
- Pre-trained on large code corpus (GitHub Open Source projects)
- Understands code semantics better than general-purpose BERT
- Good starting point for smart contract analysis
- HuggingFace hosted: microsoft/codebert-base

The model will be fine-tuned on our vulnerability dataset using causal language
modeling + layer-wise learning rate decay for efficient training.
"""

print("\n" + "="*60)
print("STEP 3: LOAD MODEL & TOKENIZER")
print("="*60)

model_name = "microsoft/codebert-base"
print(f"Loading pre-trained model: {model_name}")
print(f"  This is a BERT model with ~125M parameters")
print(f"  Pre-trained on code from GitHub")

try:
    # Load tokenizer (converts text → token IDs)
    print(f"\n✓ Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    print(f"  Vocabulary size: {tokenizer.vocab_size}")
    print(f"  Max token length: 512")
    
    # Load pre-trained model and add classification head
    print(f"\n✓ Loading model for sequence classification...")
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, 
        num_labels=2  # 2 classes: safe and vulnerable
    )
    print(f"  Total parameters: {model.num_parameters():,}")
    print(f"  Classification head will output {model.num_labels} classes")
    
    # Move model to GPU if available for faster training
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    print(f"\n✓ Model moved to: {device}")
    
except Exception as e:
    print(f"\n✗ Error loading model: {e}")
    print(f"  Check internet connection and HuggingFace availability")
    raise


STEP 3: LOAD MODEL & TOKENIZER
Loading pre-trained model: microsoft/codebert-base
  This is a BERT model with ~125M parameters
  Pre-trained on code from GitHub

✓ Loading tokenizer...
  Vocabulary size: 50265
  Max token length: 512

✓ Loading model for sequence classification...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: microsoft/codebert-base
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.weight        | UNEXPECTED | 
pooler.dense.bias          | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Total parameters: 124,647,170
  Classification head will output 2 classes

✓ Model moved to: cpu


In [5]:
# === CELL 5: Tokenize Datasets ===
"""
Convert raw contract bytecode strings to token IDs that the model can process.

Tokenization Process:
1. Split code into tokens (subword pieces learned by CodeBERT)
2. Truncate to max 512 tokens (CodeBERT's max sequence length)
3. Pad shorter sequences to 512 tokens
4. Convert to token IDs and attention masks

Output for each contract:
- input_ids: [101, 2054, 2003, ...] (token indices)
- attention_mask: [1, 1, 1, ..., 0, 0] (1=token, 0=padding)
- label: 0-3 (vulnerability class)
"""

print("\n" + "="*60)
print("STEP 4: TOKENIZE DATASETS")
print("="*60)

def tokenize_function(examples):
    """
    Tokenize contract code using CodeBERT tokenizer.
    
    Args:
        examples: Dictionary with 'code' field from dataset
    
    Returns:
        Dictionary with 'input_ids' and 'attention_mask'
    """
    return tokenizer(
        examples['code'], 
        truncation=True,           # Truncate to max_length
        padding='max_length',      # Pad shorter sequences
        max_length=512             # CodeBERT max sequence length
    )

print(f"\nTokenizing training set ({len(train_dataset)} contracts)...")
train_dataset = train_dataset.map(tokenize_function, batched=True, batch_size=32)
print(f"✓ Tokenized training set")

print(f"\nTokenizing validation set ({len(val_dataset)} contracts)...")
val_dataset = val_dataset.map(tokenize_function, batched=True, batch_size=32)
print(f"✓ Tokenized validation set")

# Set PyTorch tensor format for efficient training
print(f"\n✓ Setting PyTorch tensor format...")
train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
val_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

print(f"✓ Datasets ready for training")


STEP 4: TOKENIZE DATASETS

Tokenizing training set (196 contracts)...


Map:   0%|          | 0/196 [00:00<?, ? examples/s]

✓ Tokenized training set

Tokenizing validation set (50 contracts)...


Map:   0%|          | 0/50 [00:00<?, ? examples/s]

✓ Tokenized validation set

✓ Setting PyTorch tensor format...
✓ Datasets ready for training


In [6]:
# === CELL 6: Define Evaluation Metrics ===
"""
Compute evaluation metrics during training.

Metrics:
- Accuracy: Proportion of correct predictions
- Precision: True positives / (true positives + false positives)
- Recall: True positives / (true positives + false negatives)
- F1: Harmonic mean of precision and recall

We use weighted averaging to handle class imbalance (more 'safe' contracts
than vulnerable ones).
"""

print("\n" + "="*60)
print("STEP 5: DEFINE EVALUATION METRICS")
print("="*60)

def compute_metrics(eval_pred):
    """
    Compute precision, recall, F1, and accuracy during evaluation.
    
    Args:
        eval_pred: Tuple of (predictions, labels) from model
                   - predictions: Raw logits from model (batch_size, num_labels)
                   - labels: Ground truth labels (batch_size,)
    
    Returns:
        Dictionary with accuracy, F1, precision, recall
    """
    predictions, labels = eval_pred
    
    # Convert logits to predicted class IDs  
    predictions = np.argmax(predictions, axis=1)
    
    # Handle edge case where we have < 2 classes in a batch
    unique_classes = np.unique(labels)
    average_type = 'weighted'  # Use weighted average for imbalanced classes
    
    # Compute metrics with zero_division=0 to handle missing classes
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, 
        average=average_type, 
        zero_division=0
    )
    
    # Overall accuracy
    accuracy = accuracy_score(labels, predictions)
    
    return {
        'accuracy': accuracy, 
        'f1': f1, 
        'precision': precision, 
        'recall': recall
    }

print("✓ Metrics function configured")
print("  - Accuracy: Overall correct predictions")
print("  - Precision: How many predicted positives are actually positive")
print("  - Recall: How many actual positives were caught")
print("  - F1: Balance between precision and recall")


STEP 5: DEFINE EVALUATION METRICS
✓ Metrics function configured
  - Accuracy: Overall correct predictions
  - Precision: How many predicted positives are actually positive
  - Recall: How many actual positives were caught
  - F1: Balance between precision and recall


In [7]:
import transformers
print(transformers.__version__)

5.3.0


In [8]:
import transformers
print("Transformers version:", transformers.__version__)

Transformers version: 5.3.0


In [9]:
import sys
print(sys.executable)

c:\Users\phili\Documents\Projects\TrustChainAi\.venv\Scripts\python.exe


In [10]:
import os
from pathlib import Path
from transformers import TrainingArguments, Trainer
import shutil
import torch

print("\n" + "="*60)
print("STEP 6: TRAINING CONFIGURATION (FRESH TRAINING, FIXED)")
print("="*60)

# ------------------------------------------------------------
# 1. Setup directories
# ------------------------------------------------------------
output_dir = project_root / "data" / "models" / "checkpoints"
logs_dir = project_root / "logs"

# Remove old checkpoints for fresh training
if output_dir.exists():
    print(f"⚠ Clearing old checkpoints in: {output_dir}")
    shutil.rmtree(output_dir)
output_dir.mkdir(parents=True, exist_ok=True)
logs_dir.mkdir(parents=True, exist_ok=True)

# Set TensorBoard logging directory via env variable
os.environ["TENSORBOARD_LOGGING_DIR"] = str(logs_dir)
print(f"Model checkpoints will be saved in: {output_dir}")
print(f"TensorBoard logs will be saved in: {logs_dir}")

# ------------------------------------------------------------
# 2. Training hyperparameters (CPU-friendly, compatible with v5.x)
# ------------------------------------------------------------
training_args = TrainingArguments(
    output_dir=str(output_dir),
    save_total_limit=3,
    num_train_epochs=5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=50,
    logging_steps=10,

    # Updated argument names for v5.x
    eval_strategy="steps",   # instead of evaluation_strategy
    save_strategy="steps",

    eval_steps=20,
    save_steps=20,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    do_eval=True,
    seed=42,
)

# ------------------------------------------------------------
# 3. Training statistics
# ------------------------------------------------------------
print("\nTraining Summary")
print("-"*40)
print(f"Training samples : {len(train_dataset)}")
print(f"Validation samples : {len(val_dataset)}")
print(f"Epochs : {training_args.num_train_epochs}")
print(f"Batch size : {training_args.per_device_train_batch_size}")

# ------------------------------------------------------------
# 4. Initialize Trainer
# ------------------------------------------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)
print("✓ Trainer initialized successfully")

# ------------------------------------------------------------
# 5. Start Training
# ------------------------------------------------------------
print("\n" + "="*60)
print("STARTING FRESH TRAINING")
print("="*60)

trainer.train()
print("\n✓ Training complete!")

# ------------------------------------------------------------
# 6. Save final model
# ------------------------------------------------------------
final_model_path = project_root / "data" / "models" / "trustchain_model"
trainer.save_model(final_model_path)
print(f"\nFinal model saved to: {final_model_path}")


STEP 6: TRAINING CONFIGURATION (FRESH TRAINING, FIXED)
⚠ Clearing old checkpoints in: c:\Users\phili\Documents\Projects\TrustChainAi\data\models\checkpoints
Model checkpoints will be saved in: c:\Users\phili\Documents\Projects\TrustChainAi\data\models\checkpoints
TensorBoard logs will be saved in: c:\Users\phili\Documents\Projects\TrustChainAi\logs

Training Summary
----------------------------------------
Training samples : 196
Validation samples : 50
Epochs : 5
Batch size : 4
✓ Trainer initialized successfully

STARTING FRESH TRAINING


c:\Users\phili\Documents\Projects\TrustChainAi\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
20,0.678406,0.570721,0.840000,0.766957,0.705600,0.840000
40,0.321421,0.422937,0.840000,0.766957,0.705600,0.840000
60,0.295476,0.196759,0.960000,0.957608,0.961818,0.960000
80,0.495123,0.202284,0.960000,0.957608,0.961818,0.960000
100,0.250753,0.197538,0.960000,0.957608,0.961818,0.960000
120,0.206206,0.190760,0.960000,0.957608,0.961818,0.960000
140,0.316221,0.234246,0.940000,0.941403,0.943957,0.940000
160,0.066743,0.210374,0.940000,0.938353,0.938073,0.940000
180,0.153420,0.438252,0.900000,0.906043,0.920280,0.900000
200,0.280785,0.260899,0.960000,0.957608,0.961818,0.960000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\phili\Documents\Projects\TrustChainAi\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


✓ Training complete!


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Final model saved to: c:\Users\phili\Documents\Projects\TrustChainAi\data\models\trustchain_model


In [ ]:
# ============================================================
# CELL 8: Final Evaluation & Model Saving
# TrustChainAI – Smart Contract Vulnerability Detector
# ============================================================

import json
import numpy as np
import pandas as pd
import torch
from pathlib import Path

print("\n" + "="*60)
print("STEP 7: FINAL EVALUATION & MODEL SAVING")
print("="*60)

# ------------------------------------------------------------
# 1. Check which model checkpoint is being used
# ------------------------------------------------------------

best_checkpoint = None
if hasattr(trainer, "state") and trainer.state.best_model_checkpoint:
    best_checkpoint = trainer.state.best_model_checkpoint
    print(f"✓ Using best checkpoint: {best_checkpoint}")
else:
    print("✓ Using final trained model")

# ------------------------------------------------------------
# 2. Run final evaluation (safe method using predict())
# ------------------------------------------------------------

print("\n✓ Running final evaluation on validation set...")

pred_output = trainer.predict(val_dataset)
eval_results = pred_output.metrics

print("\n" + "="*60)
print("FINAL METRICS (VALIDATION SET)")
print("="*60)

for name, value in sorted(eval_results.items()):
    if isinstance(value, (float, int)):
        if 0 < value < 1:
            print(f"{name:20} {value*100:6.2f}%")
        else:
            print(f"{name:20} {value:10.4f}")

# ------------------------------------------------------------
# 3. Define model output directory
# ------------------------------------------------------------

model_output_dir = project_root / "data" / "models" / "vulnerability_detector_v1"
model_output_dir.mkdir(parents=True, exist_ok=True)

print(f"\n✓ Saving model artifacts to:")
print(f"  {model_output_dir}")

# ------------------------------------------------------------
# 4. Save trained model
# ------------------------------------------------------------

model.save_pretrained(str(model_output_dir))
print("  ├─ pytorch_model.bin")

# ------------------------------------------------------------
# 5. Save tokenizer
# ------------------------------------------------------------

tokenizer.save_pretrained(str(model_output_dir))
print("  ├─ tokenizer.json")
print("  └─ special_tokens_map.json")

# ------------------------------------------------------------
# 6. Save label mappings
# ------------------------------------------------------------

import json

label_map = {
    "safe": 0,
    "vulnerable": 1
}

label_map_path = model_output_dir / "label_map.json"

reverse_map = {v: k for k, v in label_map.items()}

with open(label_map_path, "w") as f:
    json.dump(
        {
            "label_to_vulnerability": reverse_map,
            "vulnerability_to_label": label_map
        },
        f,
        indent=2
    )

print("✓ Label mapping saved")

# ------------------------------------------------------------
# 7. Save training metadata
# ------------------------------------------------------------

config_output = {
    "training_metadata": {
        "model_name": "microsoft/codebert-base",
        "training_date": str(pd.Timestamp.now()),
        "num_training_examples": len(train_dataset),
        "num_validation_examples": len(val_dataset),
        "best_checkpoint": best_checkpoint,
    },
    "model_config": {
        "num_labels": len(label_map),
        "max_length": 512,
        "problem_type": "single_label_classification",
    },
    "training_results": {
        "loss": float(eval_results.get("test_loss", 0)),
        "accuracy": float(eval_results.get("test_accuracy", 0)),
        "f1": float(eval_results.get("test_f1", 0)),
        "precision": float(eval_results.get("test_precision", 0)),
        "recall": float(eval_results.get("test_recall", 0)),
    },
}

config_path = model_output_dir / "training_config.json"

with open(config_path, "w") as f:
    json.dump(config_output, f, indent=2)

print("  └─ training_config.json")

# ------------------------------------------------------------
# 8. Show sample predictions
# ------------------------------------------------------------

print("\n" + "="*60)
print("SAMPLE PREDICTIONS")
print("="*60)

sample_indices = list(range(min(3, len(val_dataset))))

sample_subset = torch.utils.data.Subset(val_dataset, sample_indices)
sample_preds = trainer.predict(sample_subset)

reverse_map = {v: k for k, v in label_map.items()}

for i, idx in enumerate(sample_indices):

    logits = sample_preds.predictions[i]
    pred_id = int(np.argmax(logits))
    confidence = float(np.max(np.exp(logits) / np.sum(np.exp(logits))))

    true_id = val_dataset[idx]["label"]

    true_id = int(val_dataset[idx]["label"])
    pred_id = int(pred_id)

    true_vuln = reverse_map[true_id]
    pred_vuln = reverse_map[pred_id]

    match = "✓" if true_id == pred_id else "✗"

    print(
        f"{match} True: {true_vuln:15} | "
        f"Pred: {pred_vuln:15} | "
        f"Confidence: {confidence:.1%}"
    )

# ------------------------------------------------------------
# 9. Completion message
# ------------------------------------------------------------

print("\n" + "="*60)
print("TRAINING COMPLETE ✓")
print("="*60)

print("\nModel saved to:")
print(f"  {model_output_dir}")

print("\nNext steps:")
print("1. Update vulnerability_detector.py to load the model")
print("2. Test smart contract scanning")
print("3. Launch Streamlit dashboard")

print("="*60)


STEP 7: FINAL EVALUATION & MODEL SAVING
✓ Using best checkpoint: c:\Users\phili\Documents\Projects\TrustChainAi\data\models\checkpoints\checkpoint-60

✓ Running final evaluation on validation set...


c:\Users\phili\Documents\Projects\TrustChainAi\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



FINAL METRICS (VALIDATION SET)
test_accuracy         96.00%
test_f1               95.76%
test_loss             19.59%
test_precision        96.18%
test_recall           96.00%
test_runtime            13.3140
test_samples_per_second     3.7550
test_steps_per_second  97.60%

✓ Saving model artifacts to:
  c:\Users\phili\Documents\Projects\TrustChainAi\data\models\vulnerability_detector_v1


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ├─ pytorch_model.bin
  ├─ tokenizer.json
  └─ special_tokens_map.json
✓ Label mapping saved
  └─ training_config.json

SAMPLE PREDICTIONS


c:\Users\phili\Documents\Projects\TrustChainAi\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


KeyError: tensor(1)

In [ ]:
# === CELL 9: Checkpoint Management ===
"""
Check training progress and manage checkpoints.
This cell helps you monitor training status and resume if needed.
"""

print("\n" + "="*60)
print("CHECKPOINT STATUS")
print("="*60)

# Check for saved checkpoints
checkpoints = list(output_dir.glob("checkpoint-*"))
if checkpoints:
    print(f"✓ Found {len(checkpoints)} checkpoints:")
    for ckpt in sorted(checkpoints, key=lambda x: int(x.name.split('-')[1])):
        step_num = ckpt.name.split('-')[1]
        print(f"  {ckpt.name} (step {step_num})")

    # Show training progress
    latest_checkpoint = sorted(checkpoints, key=lambda x: int(x.name.split('-')[1]))[-1]
    step_completed = int(latest_checkpoint.name.split('-')[1])
    progress_percent = (step_completed / total_training_steps) * 100
    print(f"\n📊 Training Progress: {step_completed}/{total_training_steps} steps ({progress_percent:.1f}%)")

    # Check if training is complete
    if step_completed >= total_training_steps:
        print("🎉 Training appears to be complete!")
    else:
        remaining_steps = total_training_steps - step_completed
        print(f"⏳ {remaining_steps} steps remaining")

else:
    print("❌ No checkpoints found - training hasn't started or was interrupted")

print(f"\n💾 Checkpoint directory: {output_dir}")

# Instructions for resuming
print("\n" + "="*60)
print("TO RESUME TRAINING:")
print("="*60)
print("1. If training was interrupted, simply re-run Cell 8")
print("2. The notebook will automatically detect and resume from the latest checkpoint")
print("3. Training will continue from where it left off")
print("4. New checkpoints will be saved every 500 steps")
print("="*60)